In [1]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from typing import List, Tuple
from typing import Dict

In [2]:
output_path = "2_Model/f0_OSMOSYS_ECU_Output.csv"
df = pd.read_csv(output_path)
planmicc = pd.read_csv("docs/PLANMICCOutput.csv")
ndc_agro = pd.read_csv("docs/NDCAgriculturaOutput.csv", dtype={"Fuel":str, "Emission":str})
ndc_uscuss = pd.read_csv("docs/NDCUSCUSSOutput.csv", dtype={"Fuel":str, "Emission":str, "Sector":str, "SpecificSector":str})

In [3]:
initial_year = 2010
final_year = 2035

In [4]:
# Filter only co2e emissions 
def get_model_emissions(dataFrame: pd.DataFrame, emissions:List[str], model_name:str) -> pd.DataFrame:
    emission_filters = (dataFrame['Year'] >= initial_year) & (dataFrame['Year'] <= final_year) & (~dataFrame['Emission'].isna()) & (~dataFrame['AnnualEmissions'].isna()) & (dataFrame['Emission'].isin(emissions))
    emissions_output = dataFrame.loc[emission_filters, ['Strategy', 'Emission','Year', 'AnnualEmissions']]
    missing_emissions =  set(emissions) - set(emissions_output['Emission'].unique())
    if len(missing_emissions) > 0:
        print("WARNING: The following emissions are missing:")
        print(missing_emissions)
    if model_name != "ndc":
        emissions_output['AnnualEmissions'] = emissions_output['AnnualEmissions'] * 1000
    emissions_df = emissions_output.pivot_table(index = ['Year'], columns='Strategy' ,values = 'AnnualEmissions', aggfunc='sum').reset_index()
    return emissions_df

In [5]:
def draw_model_emission_scatter(name:str, dataFrame:pd.DataFrame) -> List[go.Scatter]:
    data = []
    for scenario in dataFrame.columns[1:]:
        data.append(go.Scatter(name=f"{scenario}({name})", x=dataFrame['Year'], y=dataFrame[scenario]))
    
    return data

def draw_emissions_trayectory(emissions:List[Tuple[str,pd.DataFrame]], title:str):
    fig_data=[]
    for model in emissions:
        fig_data += draw_model_emission_scatter(model[0], model[1])
    
    fig = go.Figure(data=fig_data)


    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":title,
        },
        title_x=0.5,
        yaxis_title = "[GgCO2e]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f}</b>
    """)

    fig.show()

In [6]:
planmicc_emissions= get_model_emissions(planmicc, [
    'co2_ferm', 
    'co2_estiercol', 
    'co2_arroz', 
    'co2_cultivos', 
    'co2_encalado', 
    'co2_urea', 
    'co2_quemas', 
    'co2_gallinas', 
    'co2_suelo_indirectas', 
    'co2_gestiondirectaestiercol', 
], "planmicc")
df_emissions = get_model_emissions(df, [
    'co2_ferm', 
    'co2_estiercol', 
    'co2_arroz', 
    'co2_cultivos', 
    'co2_encalado', 
    'co2_urea', 
    'co2_quemas', 
    'co2_gallinas', 
    'co2_suelo_indirectas', 
    'co2_gestiondirectaestiercol', 
], "current")

ndc_emsissions = get_model_emissions(ndc_agro, [
    'CO2e_AGR_DIR', 
    'CO2e_AGR_IND', 
    'CO2e_ARR', 
    'CO2e_BIO', 
    'CO2e_ENC', 
    'CO2e_URE', 
    'CO2e_GAN_GE', 
    'CO2e_GAN_GED', 
    'CO2e_GAN_GEI', 
    'CO2e_GAN_FERM', 
], "ndc")


draw_emissions_trayectory([
    ('Current', df_emissions), 
    ('PLANMICC', planmicc_emissions), 
    ('NDC', ndc_emsissions), 
],
title="Trayectoria Emisiones Agricultura y Ganadería"
)

In [7]:

planmicc_emissions= get_model_emissions(planmicc, [
    'co2_cambio_de_uso', 
    'co2_remocion', 
], "planmicc")
df_emissions = get_model_emissions(df, [
    'co2_cambio_de_uso', 
    'co2_remocion', 
], "current")

ndc_emsissions = get_model_emissions(ndc_uscuss, [
    'CO2e_CUT', 
], "ndc")

draw_emissions_trayectory([
    ('Current', df_emissions), 
    ('PLANMICC', planmicc_emissions), 
    ("NDC", ndc_emsissions)
],
title="Trayectoria Emisiones USCUSS"
)

In [8]:
current_coverage_techs = {
    "Bosque_nativo_protegido": "Bosques (Bajo esquema de protección)",
    "Bosque_nativo_sinproteger": "Bosques (Sin esquema de protección)",
    "Cultivos": "Cultivos",
    "Humedal": "Humedales",
    "Otras_tierras": "Otras Tierras",
    "Pastizales": "Pastizales",
    "Pasturas": "Pasturas",
    "Plantacion_forestal": "Plantaciones Forestales",
    "Zona_atropica": "Zona Atrópicas",
}


def get_techs_production_by_strategy(
    dataFrame: pd.DataFrame, techs: List[str], strategy: str
) -> pd.DataFrame:
    prod_filters = (
        (dataFrame["Year"] >= initial_year)
        & (dataFrame["Year"] <= final_year)
        & (~dataFrame["Technology"].isna())
        & (~dataFrame["ProductionByTechnology"].isna())
        & (dataFrame["Technology"].isin(techs))
        & (dataFrame["Strategy"] == strategy)
    )

    prod_output = dataFrame.loc[
        prod_filters, ["Technology", "Year", "ProductionByTechnology"]
    ]

    missing_emissions = set(techs) - set(prod_output["Technology"].unique())
    if len(missing_emissions) > 0:
        print("WARNING: The following technologies are missing:")
        print(missing_emissions)

    prod = (
        prod_output.groupby(["Technology", "Year"])[["ProductionByTechnology"]]
        .agg("sum")
        .reset_index()
    )
    return prod


def draw_techs_production_area_scatter(
    prod_df: pd.DataFrame, tech_labels: Dict[str, str]
) -> List[go.Scatter]:
    data = []
    for tech in tech_labels.keys():
        tech_prod_df = prod_df.loc[
            prod_df["Technology"] == tech, ["Year", "ProductionByTechnology"]
        ]
        data.append(
            go.Scatter(
                x=tech_prod_df["Year"],
                y=tech_prod_df["ProductionByTechnology"],
                mode="lines",
                stackgroup="one",
                name=tech_labels[tech],
            )
        )
    return data


def draw_prod_coverage(
    model_df: pd.DataFrame, strategy: str, techs_labels: Dict[str, str], title: str
):
    prod_df = get_techs_production_by_strategy(
        model_df, list(techs_labels.keys()), strategy
    )
    fig_data = draw_techs_production_area_scatter(prod_df, techs_labels)
    fig = go.Figure(data=fig_data)

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor="lightgrey",
    )

    fig.update_layout(
        plot_bgcolor="white",
        title={
            "text": title,
        },
        title_x=0.5,
        yaxis_title="Área [Mha]",
    )

    fig.update_traces(
        hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """
    )

    fig.update_layout(
        legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="right", x=1)
    )

    fig.show()

def draw_model_prod_coverage(model_df:pd.DataFrame, techs_labels:Dict[str,str]):
    for strategy in model_df['Strategy'].unique():
        draw_prod_coverage(model_df, strategy, techs_labels, f'Distribución de la Cobertura Nacional - {strategy}')

In [9]:
draw_model_prod_coverage(df, current_coverage_techs)

In [10]:
abs_emi_techs = {"co2_remocion": "Absorciones", "co2_cambio_de_uso": "Emisiones"}

def draw_emissions_area_scatter(model_df:pd.DataFrame, emissions:List[str], strategy:str, model_name:str) -> List[go.Scatter]:
    data = []
    for emission in emissions:
        emission_df = get_model_emissions(model_df, [emission], model_name)
        data.append(
            go.Scatter(
                x=emission_df['Year'],
                y=emission_df[strategy],
                mode="lines",
                stackgroup=str(emission),
                name=abs_emi_techs[emission],
            )
        )

    return data

def draw_emissions_abs_emi(model_df:pd.DataFrame,abs_emi: Dict[str,str],model_name:str, strategy:str, title:str):

    abs_emi_scatters = draw_emissions_area_scatter(model_df, list(abs_emi.keys()), strategy, model_name)
    emissions_df = get_model_emissions(model_df, list(abs_emi.keys()), model_name)
    strategy_emission_df = emissions_df[['Year', strategy]].copy()
    emission_scatters = draw_model_emission_scatter(model_name,strategy_emission_df)

    fig_data = abs_emi_scatters + emission_scatters
    
    fig = go.Figure(data=fig_data)


    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":title,
        },
        title_x=0.5,
        yaxis_title = "[GgCO2e]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f}</b>
    """)

    fig.show()

def draw_model_emissions_and_absorptions(model_df:pd.DataFrame, abs_emi:Dict[str,str], model_name:str):
    for strategy in model_df['Strategy'].unique():
        draw_emissions_abs_emi(model_df, abs_emi, model_name ,strategy, f'Proyección de emisiones y absorciones de carbono del Sector USCUSS - {strategy}')

In [11]:
draw_model_emissions_and_absorptions(df, abs_emi_techs, "current")

In [12]:
agro_techs_hist = {
    "co2_ferm": "Fermentación Entérica de Ganado",
    "co2_estiercol": "Indirectas de la Gestión de Estiércol de Bovinos",
    "co2_arroz": "Producción de Arroz",
    "co2_cultivos": "Directas de Suelos Agrícolas",
    "co2_encalado": "Encalado",
    "co2_urea": "Fertilización de Urea",
    "co2_quemas": "Quema de Cultivos",
    "co2_gallinas": "Gestión de Estiércol de Aves de Corral",
    "co2_suelo_indirectas": "Indirectas de Suelos Agrícolas",
    "co2_gestiondirectaestiercol": "Directas de la Gestión de Estiércol de Bovinos",
}


def get_emissions_by_strategy(
    model_df: pd.DataFrame, emissions: List[str], strategy: str
):
    emission_filters = (
        (model_df["Year"] >= initial_year)
        & (model_df["Year"] <= final_year)
        & (~model_df["Emission"].isna())
        & (~model_df["AnnualEmissions"].isna())
        & (model_df["Emission"].isin(emissions))
        & (model_df["Strategy"] == strategy)
    )
    emissions_output = model_df.loc[
        emission_filters, ["Emission", "Year", "AnnualEmissions"]
    ]

    emissions_output['AnnualEmissions'] = emissions_output["AnnualEmissions"] * 1000

    missing_emissions = set(emissions) - set(emissions_output["Emission"].unique())

    if len(missing_emissions) > 0:
        print("WARNING: The following emissions are missing:")
        print(missing_emissions)

    emissions_df = (
        emissions_output.groupby(["Emission", "Year"]).aggregate("sum").reset_index()
    )
 
    return emissions_df


def get_histogram_bars(
    hist_df: pd.DataFrame,
    labels: Dict[str, str],
    key_column: str,
    x_column: str,
    y_column: str,
):
    bar_scatters = []
    for key in labels.keys():
        item_df = hist_df.loc[hist_df[key_column] == key]
        bar_scatters.append(
            go.Bar(name=labels[key], x=item_df[x_column], y=item_df[y_column]),
        )

    return bar_scatters


def draw_emissions_histogram_by_strategy(model_df: pd.DataFrame, strategy: str, years: List[int], emission_labels: Dict[str, str],title:str):
    emissions_df = get_emissions_by_strategy(
        model_df, list(emission_labels.keys()), strategy
    )
    hist_df = emissions_df.loc[emissions_df["Year"].isin(years)].copy()
    hist_df['Year'] = hist_df['Year'].astype(int).astype(str)
    total_df = hist_df.groupby('Year')['AnnualEmissions'].aggregate('sum').reset_index()
    fig_data = get_histogram_bars(hist_df, emission_labels, key_column="Emission", x_column="Year", y_column="AnnualEmissions")

    # Fig. Emissions per Technology vs Year
    fig = go.Figure(
        data=fig_data,
        layout=go.Layout(barmode="stack"),
    )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor="lightgrey",
    )

    fig.update_layout(
        plot_bgcolor="white",
        title=title,
        title_x=0.5,
        yaxis_title="[GgCO2e]",
    )

    fig.update_traces(
        hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """
    )

    fig.add_trace(go.Scatter(mode='text',
                        x=total_df['Year'],
                        y=total_df['AnnualEmissions'],
                        textposition='top center',
                        text=[str(x) for x in total_df['AnnualEmissions'].tolist()],
                        texttemplate="%{y:.3f}",
                        showlegend=False,
                        hoverinfo='skip'
                    ))
    fig.show()

def draw_model_emissions_histogram(model_df:pd.DataFrame, emission_labels:Dict[str,str]):
    for strategy in model_df['Strategy'].unique():
        draw_emissions_histogram_by_strategy(model_df, strategy, [2010,2018,2035,2050,2070], emission_labels, f'Emisiones Escenario {strategy} para el sector Agricultura')

In [13]:
draw_model_emissions_histogram(df, agro_techs_hist)